<a href="https://colab.research.google.com/github/rselvak6/opendde-colab/blob/main/opendde_monomer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDDE on Colab

Requires a GPU runtime: **suggested: A100 high-ram**

This notebook: installs OpenDDE from source with `uv`, downloads model weights + inference data, runs a tiny smoke test, then runs your real prediction job.

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

downloading uv 0.13.6 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'OpenDDE'...
remote: Enumerating objects: 1008, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 1008 (delta 209), reused 197 (delta 197), pack-reused 747 (from 1)
Receiving objects: 100% (1008/1008), 27.46 MiB | 20.48 MiB/s, done.
Resolving deltas: 100% (573/573), done.
/content/OpenDDE
Using Python 3.13.16 environment at: /usr
Resolved 66 packages in 1.38s
Prepared 31 packages in 25.10s
Uninstalled 18 packages in 858ms
Installed 31 packages in 342ms
 + biopython==1.85
 + biotite==1.4.0
 + biotraj==1.2.2
 + cuequivariance==0.10.0
 + cuequivariance-ops-cu12==0.10.0
 + cuequivariance-ops-torch-cu12==0.10.0
 + cuequivariance-torch==0.10.0
 + ml-collections==1.1.0
 - numpy==2.1.3
 + numpy==2.4.1
 - nvidia-cublas-cu12==12.9.2.10
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.9.79
 + nvi

In [2]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

env: OPENDDE_ROOT_DIR=/content/data


In [3]:
# Download only what this notebook actually uses.
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
[INFO] Using OPENDDE_ROOT_DIR: /content/data
[INFO] Downloading common inference data files from https://huggingface.co/aurekaresearch/OpenDDE/resolve/eddd563ce96571f784012edd8f045181c8f8627d/common
[INFO] Downloading https://huggingface.co/aurekaresearch/OpenDDE/resolve/eddd563ce96571f784012edd8f045181c8f8627d/common/components.cif
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  S

In [4]:
import os
os.environ["OPENDDE_FORCE_SAMPLE_DIFFUSION_AMP"] = "1"
os.environ["OPENDDE_FORCE_CONFIDENCE_AMP"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [9]:
# Must match the "name" field of the input JSON: OpenDDE writes results to
# ./output/<name>/, which every downstream cell reads, and the run cell below
# loads /content/OpenDDE/<PROJECT_NAME>.json. Pick a known input or type one.
PROJECT_NAME = 'egfr-rad51-monomer'  # @param ["egfr-rad51-monomer", "egfr-rad51-monomer-afatinib", "egfr-rad51-monomer-alflutinib", "egfr-rad51-monomer-bh30643", "egfr-rad51-monomer-erlotinib", "egfr-rad51-monomer-gefitinib", "egfr-rad51-monomer-lazertinib", "egfr-rad51-monomer-osimertinib"] {allow-input: true}

In [10]:
# @title Pick which uploaded projects to run
# Lists every OpenDDE input JSON in /content/OpenDDE, validates it, and offers a
# multi-select. Files the pipeline generates are hidden rather than offered as
# projects: the *-tactive / *-tinactive inputs the staging cell writes, and the
# *-update-msa.json OpenDDE writes during its MSA step.
#
# This cell and the next are the BATCH path. They rebuild RUNS for every
# selected project x every template, replacing what the single-project
# "Template sweep setup" cell above put there. Run that cell once first anyway -
# it defines the staging helpers these two use - then run these.
#
# PROJECTS_OVERRIDE wins over the widget when it is non-empty: "all", or a
# comma-separated list of project names. Useful if the widget does not render.
PROJECTS_OVERRIDE = ""  # @param {type:"string"}

import glob
import json
import os

INPUTS_DIR = globals().get("INPUT_DIR", "/content/OpenDDE")
GENERATED_SUFFIXES = ("-tactive", "-tinactive", "-update-msa", "-reusemsa")


def _inspect(path):
    """Returns (job, problem). job is None when the file is unusable."""
    try:
        with open(path) as f:
            jobs = json.load(f)
    except Exception as exc:
        return None, "not valid JSON (%s)" % exc
    if not isinstance(jobs, list) or len(jobs) != 1:
        return None, "expected a list holding exactly one job"
    job = jobs[0]
    if not isinstance(job, dict) or "name" not in job or "sequences" not in job:
        return None, "missing \"name\" or \"sequences\""
    stem = os.path.splitext(os.path.basename(path))[0]
    if job["name"] != stem:
        # OpenDDE writes results to ./output/<name> while every cell here keys
        # off the file stem, so a mismatch would silently analyse the wrong run.
        return None, ("\"name\" is %r but the file is %s.json; OpenDDE writes to "
                      "./output/<name>, so they must match" % (job["name"], stem))
    if not [e for e in job["sequences"] if "proteinChain" in e]:
        return None, "no proteinChain, so no template could be attached"
    return job, None


candidates = sorted(glob.glob(os.path.join(INPUTS_DIR, "*.json")))
hidden, valid, broken = [], [], []
for p in candidates:
    stem = os.path.splitext(os.path.basename(p))[0]
    if stem.endswith(GENERATED_SUFFIXES):
        hidden.append(stem)
        continue
    job, problem = _inspect(p)
    if problem:
        broken.append((stem, problem))
    else:
        n_chain = sum(e["proteinChain"].get("count", 1)
                      for e in job["sequences"] if "proteinChain" in e)
        n_lig = sum(1 for e in job["sequences"] if "ligand" in e or "ion" in e)
        valid.append((stem, n_chain, n_lig, len(job.get("modelSeeds") or [])))

print("Input JSONs in %s" % INPUTS_DIR)
print()
if valid:
    print("  %-40s %-8s %-8s %s" % ("project", "chains", "ligands", "seeds"))
    print("  " + "-" * 68)
    for stem, n_chain, n_lig, n_seed in valid:
        print("  %-40s %-8d %-8d %d" % (stem, n_chain, n_lig, n_seed))
else:
    print("  (none usable)")
if broken:
    print()
    print("Skipped as unusable:")
    for stem, problem in broken:
        print("  %-40s %s" % (stem, problem))
if hidden:
    print()
    print("Hidden (generated by the pipeline): %s" % ", ".join(sorted(hidden)))

names = [v[0] for v in valid]
PROJECT_PICKER = None
SELECTED_PROJECTS = []

override = PROJECTS_OVERRIDE.strip()
if override:
    if override.lower() == "all":
        SELECTED_PROJECTS = list(names)
    else:
        wanted = [w.strip() for w in override.split(",") if w.strip()]
        unknown = [w for w in wanted if w not in names]
        if unknown:
            raise RuntimeError(
                "PROJECTS_OVERRIDE names projects that are not usable inputs here: "
                "%s\nAvailable: %s" % (", ".join(unknown), ", ".join(names) or "none"))
        SELECTED_PROJECTS = wanted
    print()
    print("PROJECTS_OVERRIDE in force - %d project(s) selected, the widget is "
          "ignored:" % len(SELECTED_PROJECTS))
    for n in SELECTED_PROJECTS:
        print("   ", n)
elif not names:
    print()
    print("Nothing to select. Upload your input JSONs to %s first." % INPUTS_DIR)
else:
    try:
        import ipywidgets as widgets
        from IPython.display import display

        PROJECT_PICKER = widgets.SelectMultiple(
            options=names,
            value=tuple(names),          # everything pre-selected
            rows=min(14, max(3, len(names))),
            description="Projects",
            layout=widgets.Layout(width="80%"),
        )
        print()
        print("All %d pre-selected. Ctrl/Cmd-click or shift-click to narrow it, "
              "then run the next cell." % len(names))
        display(PROJECT_PICKER)
    except Exception as exc:
        print()
        print("ipywidgets unavailable (%s). Put the projects you want in "
              "PROJECTS_OVERRIDE above (or \"all\") and re-run this cell." % exc)

Input JSONs in /content/OpenDDE

  project                                  chains   ligands  seeds
  --------------------------------------------------------------------
  egfr-rad51-monomer-alflutinib            1        1        25
  egfr-rad51-monomer-bh30643               1        1        25
  egfr-rad51-monomer-erlotinib             1        1        25
  egfr-rad51-monomer-gefitinib             1        1        25
  egfr-rad51-monomer-lazertinib            1        1        25
  egfr-rad51-monomer-osimertinib           1        1        25
  egfr-rad51-monomer                       1        0        25

All 7 pre-selected. Ctrl/Cmd-click or shift-click to narrow it, then run the next cell.


SelectMultiple(description='Projects', index=(0, 1, 2, 3, 4, 5, 6), layout=Layout(width='80%'), options=('egfr…

In [7]:
# @title Template sweep setup: stage both supplied templates for PROJECT_NAME
# Set PROJECT_NAME in the cell above and run this; it prepares every template
# variant of that one compound in a single pass. The next cell then predicts
# all three. Nothing else needs changing per variant.
#
# The no-template variant keeps the bare PROJECT_NAME as its run name, so runs
# predicted before templates existed are reused rather than repeated.
# OpenDDE does NOT take a structure file as a template input. Its only template
# input is "templatesPath" on a proteinChain, which must point at an hmmsearch
# .a3m (or HHsearch .hhr) *hit* file; the structure itself is then looked up by
# the 4-character PDB ID in the hit name, under
# $OPENDDE_ROOT_DIR/search_database/mmcif/<pdb_id>.cif (falling back to a PDBe
# download for IDs it does not find locally). So to use a hand-built template we
# register it as if it were a PDB entry. This cell does that end to end.
#
# Four gates have to be satisfied, and three of them fail SILENTLY - the run
# just reports "Found 0 templates" and predicts as if --use_template were false:
#
#  1. The hit name must be 4 alphanumerics + "_" + chain
#     (template_parser.get_pdb_id_and_chain). We use IDs starting with a letter:
#     every one of the 219,912 real PDB IDs in release_date_cache.json starts
#     with a digit, so "egfa"/"egfi" can never collide with a real entry.
#  2. The ID must be present in $OPENDDE_ROOT_DIR/common/release_date_cache.json,
#     or TemplateHitFilter.prefilter drops the hit as "<id> date unknown".
#  3. The CIF's own min(_pdbx_audit_revision_history.revision_date) must be
#     <= 2021-09-30, the checkpoint's data cutoff, hardcoded as
#     max_template_date in infer_dataloader.py. A CIF with no revision date is
#     treated as 9999-12-31 and dropped. (The supplied files already carry
#     2021-09-30, so they pass with zero margin.)
#  4. Some monomer's _chem_comp.type must contain the lowercase substring
#     "peptide" (template_parser._get_protein_chains). The PDB writes
#     "L-peptide linking"; AlphaFold3 writes "L-PEPTIDE LINKING", which fails
#     that test and yields "No protein chains found". We normalize the casing on
#     the copy we stage, leaving the originals untouched.
#
# The .a3m needs less than you would expect: process() re-aligns the query to
# the CIF's own seqres with kalign and rebuilds the hit indices
# (_update_realigned_hit), so the alignment written here is used only for the
# prefilter's coverage ratio. We still emit a true alignment so the file is
# honest and stays correct if that ever changes.
TEMPLATE_DIR = "/content/templates"  # @param {type:"string"}

import glob
import io
import json
import os
import re
import shutil
import subprocess
import tempfile

ROOT = os.environ.get("OPENDDE_ROOT_DIR", "/content/data")
MMCIF_DIR = os.path.join(ROOT, "search_database", "mmcif")
TPL_CACHE_DIR = os.path.join(ROOT, "search_database", "template_cache")
DATE_CACHE = os.path.join(ROOT, "common", "release_date_cache.json")
A3M_DIR = "/content/template_hits"
INPUT_DIR = "/content/OpenDDE"

# filename substring -> 4-char registration ID. "idealinactive" is not a
# substring of "idealactive", so these two patterns stay disjoint.
TEMPLATES = {
    "active": {"id": "egfa", "match": "idealactive",
               "note": "EGFR kinase active supplied template"},
    "inactive": {"id": "egfi", "match": "idealinactive",
                 "note": "EGFR kinase inactive supplied template"},
}
CUTOFF = "2021-09-30"
MIN_TEMPLATE_COVERAGE = 0.5  # fraction of template residues matched to a chain


def _normalize_chem_comp_type(text):
    """Lowercase "PEPTIDE" inside _chem_comp.type values (PDB casing).

    Data-driven: only the exact quoted strings that actually appear in this
    file's _chem_comp.type column are rewritten, so nothing else is touched.
    """
    from Bio.PDB.MMCIF2Dict import MMCIF2Dict

    types = set(MMCIF2Dict(io.StringIO(text)).get("_chem_comp.type", []))
    changed = []
    for t in types:
        if "PEPTIDE" in t.upper() and "peptide" not in t:
            fixed = re.sub("PEPTIDE", "peptide", t, flags=re.I)
            for q in ('"%s"' % t, "'%s'" % t):
                if q in text:
                    text = text.replace(q, q[0] + fixed + q[0])
            changed.append("%s -> %s" % (t, fixed))
    return text, changed


def _parse_template_cif(text):
    """Returns (seqres_one_letter, author_chain_id, release_date).

    Mirrors template_parser.TemplateParser.parse so a file that fails here
    would have failed inside OpenDDE too.
    """
    import collections

    from Bio import PDB
    from Bio.Data import PDBData

    p = PDB.MMCIFParser(QUIET=True)
    p.get_structure("", io.StringIO(text))
    info = dict(p._mmcif_dict)
    for k, v in list(info.items()):
        if not isinstance(v, list):
            info[k] = [v]

    def loop(prefix):
        cols = [k for k in info if k.startswith(prefix)]
        if not cols:
            return []
        return [dict(zip(cols, row)) for row in zip(*(info[c] for c in cols))]

    dates = info.get("_pdbx_audit_revision_history.revision_date") or []
    release = min(dates) if dates else ""

    polymers = collections.defaultdict(list)
    for e in loop("_entity_poly_seq."):
        polymers[e["_entity_poly_seq.entity_id"]].append(e["_entity_poly_seq.mon_id"])
    chem = {e["_chem_comp.id"]: e for e in loop("_chem_comp.")}
    ent2chain = collections.defaultdict(list)
    for a in loop("_struct_asym."):
        ent2chain[a["_struct_asym.entity_id"]].append(a["_struct_asym.id"])

    valid = {}
    for eid, mons in polymers.items():
        if any("peptide" in chem.get(m, {}).get("_chem_comp.type", "") for m in mons):
            for cid in ent2chain[eid]:
                valid[cid] = mons
    if not valid:
        raise RuntimeError("no protein chains (gate 4) - _chem_comp.type casing?")

    label2auth = {}
    for lab, auth, mdl in zip(info["_atom_site.label_asym_id"],
                              info["_atom_site.auth_asym_id"],
                              info["_atom_site.pdbx_PDB_model_num"]):
        if mdl == "1":
            label2auth[lab] = auth
    for cid, mons in valid.items():
        if cid in label2auth:
            seq = "".join(PDBData.protein_letters_3to1.get(m, "X") for m in mons)
            seq = "".join(c if len(c) == 1 else "X" for c in seq)
            return seq, label2auth[cid], release
    raise RuntimeError("no protein chain present in model 1")


def _align_blocks(query, tseq):
    from Bio import Align
    from Bio.Align import substitution_matrices

    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.open_gap_score = -11
    aligner.extend_gap_score = -1
    aligner.mode = "local"
    aln = aligner.align(query, tseq)[0]
    return [(tuple(q), tuple(t)) for q, t in zip(*aln.aligned)]


def _coverage(query, tseq, blocks):
    ident = sum(
        1
        for (qs, qe), (ts, _) in blocks
        for k in range(qe - qs)
        if query[qs + k] == tseq[ts + k]
    )
    return ident / len(tseq)


def _a3m_text(query, tseq, tpl_id, chain, note, blocks):
    """One hmmsearch-style a3m hit: query columns upper/'-', insertions lower."""
    col = ["-"] * len(query)
    ins = [""] * len(query)
    prev_q_end = prev_t_end = None
    for (qs, qe), (ts, te) in blocks:
        if prev_t_end is not None and ts > prev_t_end:
            ins[prev_q_end - 1] += tseq[prev_t_end:ts].lower()
        for k in range(qe - qs):
            col[qs + k] = tseq[ts + k].upper()
        prev_q_end, prev_t_end = qe, te
    start = blocks[0][1][0] + 1
    end = blocks[-1][1][1]
    hit = "".join(col[i] + ins[i] for i in range(len(query)))
    desc = ">%s_%s/%d-%d mol:protein length:%d  %s" % (
        tpl_id, chain, start, end, len(tseq), note)
    return desc + "\n" + hit + "\n"


def _patch_date_cache(ids):
    if not os.path.exists(DATE_CACHE):
        raise RuntimeError(
            "missing %s - run the data-download cell first (it is part of the "
            "'common' files, not the skipped search database)." % DATE_CACHE)
    with open(DATE_CACHE) as f:
        cache = json.load(f)
    added = [i for i in ids if cache.get(i, {}).get("release_date") != CUTOFF]
    if added:
        for i in added:
            cache[i] = {"release_date": CUTOFF}
        tmp = DATE_CACHE + ".tmp"
        with open(tmp, "w") as f:
            json.dump(cache, f)
        os.replace(tmp, DATE_CACHE)
    return added, len(cache)


def _check_kalign():
    exe = shutil.which("kalign")
    if exe is None:
        print("  kalign not found - installing (needed to align query->template)")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "kalign"], check=False)
        exe = shutil.which("kalign")
    if exe is None:
        raise RuntimeError("kalign still unavailable; templates cannot be used")
    # Smoke-test the exact CLI OpenDDE calls (kalign 3.x: -i / -o / -format).
    with tempfile.TemporaryDirectory() as d:
        fi, fo = os.path.join(d, "i.fa"), os.path.join(d, "o.a3m")
        with open(fi, "w") as f:
            f.write(">a\nACDEFGHIKL\n>b\nACDEFGHIKL\n")
        r = subprocess.run([exe, "-i", fi, "-o", fo, "-format", "fasta"],
                           capture_output=True, text=True)
        if r.returncode or not os.path.exists(fo):
            raise RuntimeError(
                "kalign at %s does not accept OpenDDE's '-i/-o/-format fasta' "
                "CLI (needs kalign 3.x).\nstderr: %s" % (exe, r.stderr[:400]))
    print("  kalign OK: %s" % exe)
    return exe


def stage_template(which, base_name, template_dir, input_dir=INPUT_DIR,
                   verify_kalign=True):
    """Stages one template and returns (input_json, run_name)."""
    spec = TEMPLATES[which]
    tpl_id = spec["id"]
    print("Staging %s template as PDB-like entry '%s'" % (which, tpl_id))
    if verify_kalign:
        _check_kalign()

    cands = [p for p in sorted(glob.glob(os.path.join(template_dir, "*.cif")))
             if spec["match"] in os.path.basename(p).lower()]
    if len(cands) != 1:
        raise RuntimeError(
            "expected exactly 1 *.cif matching '%s' in %s, found %d: %s\n"
            "Upload the two template CIFs to %s first."
            % (spec["match"], template_dir, len(cands),
               [os.path.basename(c) for c in cands], template_dir))
    src = cands[0]
    print("  source: %s" % os.path.basename(src))

    with open(src, encoding="utf-8") as f:
        text = f.read()
    text, changed = _normalize_chem_comp_type(text)
    for c in changed:
        print("  _chem_comp.type normalized: %s" % c)
    tseq, chain, release = _parse_template_cif(text)
    print("  chain %s, %d residues, revision_date %s" % (chain, len(tseq), release))
    if not release or release > CUTOFF:
        raise RuntimeError(
            "release date %r is missing or after the %s cutoff; the hit would be "
            "dropped silently." % (release, CUTOFF))

    os.makedirs(MMCIF_DIR, exist_ok=True)
    os.makedirs(A3M_DIR, exist_ok=True)
    staged = os.path.join(MMCIF_DIR, tpl_id + ".cif")
    with open(staged, "w", encoding="utf-8") as f:
        f.write(text)
    print("  staged CIF: %s" % staged)

    # A parsed-CIF pickle from an earlier staging of the same ID would shadow
    # the file just written.
    for p in glob.glob(os.path.join(TPL_CACHE_DIR, "%s_*.pkl" % tpl_id)):
        os.remove(p)
        print("  cleared stale parse cache: %s" % os.path.basename(p))

    added, total = _patch_date_cache([tpl_id])
    print("  release_date_cache.json: %s (%d entries)"
          % ("added " + tpl_id if added else tpl_id + " already present", total))

    base_json = os.path.join(input_dir, base_name + ".json")
    with open(base_json) as f:
        jobs = json.load(f)
    if len(jobs) != 1:
        raise RuntimeError("expected a single job in %s, got %d"
                           % (base_json, len(jobs)))
    job = jobs[0]

    attached = skipped = 0
    for idx, ent in enumerate(job["sequences"]):
        if "proteinChain" not in ent:
            continue
        chain_seq = ent["proteinChain"]["sequence"]
        blocks = _align_blocks(chain_seq, tseq)
        cov = _coverage(chain_seq, tseq, blocks)
        ids = ent["proteinChain"].get("id")
        if cov < MIN_TEMPLATE_COVERAGE:
            skipped += 1
            print("  chain(s) %s: coverage %.2f < %.2f -> no template (correct "
                  "for non-EGFR chains)" % (ids, cov, MIN_TEMPLATE_COVERAGE))
            continue
        # Keyed by project as well as template: the hit file is built against
        # THIS chain's sequence, so staging a second project must not overwrite
        # the a3m a previously written input JSON still points at.
        a3m = os.path.join(A3M_DIR, "%s_%s_e%d.a3m" % (base_name, tpl_id, idx))
        with open(a3m, "w") as f:
            f.write(_a3m_text(chain_seq, tseq, tpl_id, chain, spec["note"], blocks))
        ent["proteinChain"]["templatesPath"] = a3m
        attached += 1
        print("  chain(s) %s: coverage %.2f -> %s" % (ids, cov, a3m))
    if not attached:
        raise RuntimeError("no protein chain matched the template; nothing to do")

    run_name = "%s-t%s" % (base_name, which)
    job["name"] = run_name
    out_json = os.path.join(input_dir, run_name + ".json")
    with open(out_json, "w") as f:
        json.dump(jobs, f, indent=2)
    print("  wrote %s (%d chain(s) templated, %d skipped)"
          % (out_json, attached, skipped))
    return out_json, run_name

# ---------------------------------------------------------------- sweep
# One entry per template variant of this compound. "none" deliberately keeps
# the bare project name: its output directory is the one earlier no-template
# runs already wrote to, so the pred cell below can skip it instead of
# spending a GPU hour reproducing it.
BASE_NAME = PROJECT_NAME
RUNS = [{
    "template": "none",
    "run_name": BASE_NAME,
    "input_json": os.path.join(INPUT_DIR, BASE_NAME + ".json"),
    "use_template": "false",
}]
for _i, _which in enumerate(("active", "inactive")):
    print()
    _json, _run = stage_template(_which, BASE_NAME, TEMPLATE_DIR,
                                 verify_kalign=(_i == 0))
    RUNS.append({"template": _which, "run_name": _run,
                 "input_json": _json, "use_template": "true"})

RUNS_BY_TEMPLATE = {r["template"]: r for r in RUNS}
# PROJECT_NAME is left at the base name here: the sweep produces three runs, so
# the analysis cells are pointed at one of them by the selector cell further
# down rather than by this cell.
print()
print("=" * 72)
print("%d variants staged for %s" % (len(RUNS), BASE_NAME))
for r in RUNS:
    print("  %-9s use_template=%-5s ./output/%s" %
          (r["template"], r["use_template"], r["run_name"]))
print("=" * 72)


NameError: name 'PROJECT_NAME' is not defined

In [6]:
# @title Stage all templates for the selected projects
# Rebuilds RUNS as (every project picked above) x (none, active, inactive) and
# hands it to the prediction cell, which already loops over RUNS and needs no
# change to run a batch.
#
# Ordering is project-major: a compound's three variants are adjacent, so if the
# batch is interrupted the compounds that did finish are complete sets rather
# than three partial comparisons.
#
# As in the single-project path, the "none" variant keeps the bare project name,
# so a no-template run already on disk is reused instead of repeated, and the
# prediction cell skips any variant whose output directory already holds .cif
# files unless FORCE_RERUN is ticked.

import os

if "stage_template" not in globals():
    raise RuntimeError(
        "run the 'Template sweep setup' cell above once first - it defines "
        "stage_template and the template paths that this cell uses")

selected = list(globals().get("SELECTED_PROJECTS") or [])
if not selected and globals().get("PROJECT_PICKER") is not None:
    selected = list(PROJECT_PICKER.value)
if not selected:
    raise RuntimeError(
        "no projects selected - pick some in the cell above (or set "
        "PROJECTS_OVERRIDE there) and run it, then run this cell")

TEMPLATE_ORDER = ("none", "active", "inactive")

RUNS = []
_verified_kalign = False
for _proj in selected:
    _base_json = os.path.join(INPUT_DIR, _proj + ".json")
    if not os.path.isfile(_base_json):
        raise RuntimeError("missing input JSON: %s" % _base_json)
    for _which in TEMPLATE_ORDER:
        if _which == "none":
            RUNS.append({"template": "none", "project": _proj, "run_name": _proj,
                         "input_json": _base_json, "use_template": "false"})
            continue
        print()
        _json_path, _run_name = stage_template(
            _which, _proj, TEMPLATE_DIR, verify_kalign=not _verified_kalign)
        _verified_kalign = True
        RUNS.append({"template": _which, "project": _proj, "run_name": _run_name,
                     "input_json": _json_path, "use_template": "true"})

RUNS_BY_NAME = {r["run_name"]: r for r in RUNS}
if len(selected) == 1:
    # Keep the single-project analysis selector working unchanged.
    RUNS_BY_TEMPLATE = {r["template"]: r for r in RUNS}
else:
    # It cannot name one run per template across several projects, and a stale
    # copy would quietly point the analysis cells at the wrong compound.
    globals().pop("RUNS_BY_TEMPLATE", None)

print()
print("=" * 86)
print("%d project(s) x %d template(s) = %d run(s)"
      % (len(selected), len(TEMPLATE_ORDER), len(RUNS)))
print()
print("  %-9s %-52s %s" % ("template", "run / output directory", "already done"))
print("  " + "-" * 82)
import glob as _glob
_todo = 0
for r in RUNS:
    _n = len(_glob.glob(os.path.join("./output", r["run_name"], "**", "*.cif"),
                        recursive=True))
    if not _n:
        _todo += 1
    print("  %-9s %-52s %s" % (r["template"], r["run_name"],
                               ("%d .cif" % _n) if _n else "-"))
print("  " + "-" * 82)
print("%d of %d still to predict." % (_todo, len(RUNS)))
if globals().get("FORCE_RERUN"):
    # Fine for a single re-run, expensive across a batch: it discards the skip
    # protection for every variant, including ones already finished.
    print()
    print("!! FORCE_RERUN is currently True, so the next cell will re-predict "
          "ALL %d run(s), not just the %d outstanding. Untick it first unless "
          "you mean to redo everything." % (len(RUNS), _todo))
if len(selected) > 1:
    print()
    print("RUNS_BY_TEMPLATE was removed: with several projects it cannot name one "
          "run per template. The analysis cells read ./output/<PROJECT_NAME>, so "
          "to inspect one run in the notebook, set PROJECT_NAME to that project "
          "and re-run the single-project 'Template sweep setup' cell - the "
          "prediction cell will skip what is already on disk.")
print("=" * 86)

RuntimeError: run the 'Template sweep setup' cell above once first - it defines stage_template and the template paths that this cell uses

In [ ]:
# @title Predict every template variant of PROJECT_NAME
# One `opendde pred` per entry in RUNS. The template is selected per run by
# --use_template plus the generated input JSON (whose proteinChain carries
# templatesPath), so a single hard-coded pred line cannot cover the sweep.
#
# A variant whose output directory already holds .cif files is skipped, which
# makes the cell safe to re-run after a crash or an OOM on one variant. Tick
# FORCE_RERUN to redo them anyway.
FORCE_RERUN = True  # @param {type:"boolean"}
# Reuse a protein MSA that an earlier run already fetched, instead of
# asking api.colabfold.com again for a sequence it has already answered
# for. The time saved is small - the server answered in about 3s per chain
# in the logs - but every run then uses the SAME MSA, so differences
# between variants come from the template and the ligand rather than from
# MSA variation, and a long batch stops depending on that server being up
# for all 24 runs. Untick to let every run fetch its own.
MSA_REUSE = True  # @param {type:"boolean"}

import glob
import json
import os
import subprocess
import threading
import time

if "RUNS" not in globals():
    raise RuntimeError("run the template sweep cell above first")

# Per-variant GPU utilisation, read by the "Check GPU utilisation" cell.
# Colab runs one cell at a time, so utilisation during a prediction cannot
# be observed from another cell - it has to be sampled from inside this one.
GPU_PEAKS = globals().get("GPU_PEAKS", {})


def _sample_gpus(stop_event, peaks, period=5.0):
    """Poll nvidia-smi until stop_event is set. Never raises into the run."""
    while not stop_event.is_set():
        try:
            out = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=index,utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=30).stdout
            for line in out.strip().splitlines():
                idx, util, mem = (c.strip() for c in line.split(","))
                p = peaks.setdefault(int(idx), {"max_util": 0, "sum_util": 0,
                                                "n": 0, "max_mem": 0})
                p["max_util"] = max(p["max_util"], int(util))
                p["max_mem"] = max(p["max_mem"], int(mem))
                p["sum_util"] += int(util)
                p["n"] += 1
        except Exception:
            return  # no nvidia-smi, or it misbehaved: just stop sampling
        stop_event.wait(period)


def _msa_index():
    """{sequence: (pairedMsaPath, unpairedMsaPath)} from MSA already done.

    Built from the *-update-msa.json files OpenDDE writes beside an input
    after a server search. Keyed by sequence, so it serves a different
    template of the same project and a different compound that shares the
    protein equally well. Entries whose a3m files are missing are dropped,
    so a deleted ./output cannot leave a run pointing at nothing.
    """
    index = {}
    for p in sorted(glob.glob(os.path.join(INPUT_DIR, "*-update-msa.json"))):
        try:
            with open(p) as f:
                jobs = json.load(f)
        except Exception:
            continue
        for job in jobs if isinstance(jobs, list) else []:
            for ent in job.get("sequences") or []:
                c = ent.get("proteinChain") if isinstance(ent, dict) else None
                if not c or "sequence" not in c:
                    continue
                paths = [c.get("pairedMsaPath"), c.get("unpairedMsaPath")]
                paths = [os.path.abspath(q) for q in paths if q]
                if not paths or not all(os.path.isfile(q) for q in paths):
                    continue
                index.setdefault(c["sequence"], (
                    os.path.abspath(c["pairedMsaPath"])
                    if c.get("pairedMsaPath") else None,
                    os.path.abspath(c["unpairedMsaPath"])
                    if c.get("unpairedMsaPath") else None))
    return index


def _apply_msa_reuse(r, index):
    """Point one run at MSA files already on disk; returns a note or None."""
    if not index:
        return None
    with open(r["input_json"]) as f:
        jobs = json.load(f)
    filled = []
    for ent in jobs[0].get("sequences") or []:
        c = ent.get("proteinChain") if isinstance(ent, dict) else None
        if not c or c.get("pairedMsaPath") or c.get("unpairedMsaPath"):
            continue
        hit = index.get(c.get("sequence"))
        if not hit:
            continue
        paired, unpaired = hit
        if paired:
            c["pairedMsaPath"] = paired
        if unpaired:
            c["unpairedMsaPath"] = unpaired
        filled.append(",".join(c.get("id") or []) or "?")
    if not filled:
        return None
    if r["use_template"] == "true":
        target = r["input_json"]  # generated by staging, ours to edit
    else:
        # Never edit an uploaded input. A derived copy keeps "name"
        # untouched, so output still lands in ./output/<run_name>.
        target = os.path.join(INPUT_DIR, r["run_name"] + "-reusemsa.json")
        r["input_json"] = target
    with open(target, "w") as f:
        json.dump(jobs, f, indent=2)
    return "reusing MSA for chain(s) %s" % " ".join(filled)


PRED_FLAGS = ["-o", "./output", "-n", "opendde_v1", "--use_msa", "true",
              "--use_rna_msa", "false", "--sample", "1", "--step", "200",
              "--cycle", "10", "--need_atom_confidence", "true", "--dtype", "bf16"]

_summary = []
for r in RUNS:
    out_dir = os.path.join("./output", r["run_name"])
    existing = glob.glob(os.path.join(out_dir, "**", "*.cif"), recursive=True)
    label = "template=%s -> ./output/%s" % (r["template"], r["run_name"])
    if existing and not FORCE_RERUN:
        print("== SKIP  %s (%d .cif already there)" % (label, len(existing)))
        _summary.append((r["template"], r["run_name"], "skipped", len(existing)))
        continue
    _note = None
    if MSA_REUSE:
        try:
            # Rebuilt per run: the first run of a sequence creates the
            # update-msa.json that every later one reads.
            _note = _apply_msa_reuse(r, _msa_index())
        except Exception as _exc:
            print("   (MSA reuse skipped: %s)" % _exc)
    cmd = (["opendde", "pred", "-i", r["input_json"],
            "--use_template", r["use_template"]] + PRED_FLAGS)
    print()
    print("== RUN   %s" % label)
    print("   %s" % " ".join(cmd))
    if _note:
        print("   %s" % _note)
    print()
    _peaks = {}
    _stop = threading.Event()
    _sampler = threading.Thread(target=_sample_gpus, args=(_stop, _peaks),
                                daemon=True)
    _sampler.start()
    _t0 = time.time()
    try:
        code = subprocess.run(cmd).returncode
    finally:
        _stop.set()
        _sampler.join(timeout=10)
    GPU_PEAKS[r["run_name"]] = _peaks
    print("   %s in %.1f min%s" % (
        "finished" if code == 0 else "failed", (time.time() - _t0) / 60,
        "".join(", GPU %d peak %d%% / %d MiB"
                % (i, _peaks[i]["max_util"], _peaks[i]["max_mem"])
                for i in sorted(_peaks))))
    n = len(glob.glob(os.path.join(out_dir, "**", "*.cif"), recursive=True))
    _summary.append((r["template"], r["run_name"],
                     "ok" if code == 0 else "FAILED rc=%d" % code, n))
    if code != 0:
        # Keep going: an OOM on one variant should not cost the other two.
        print("!! %s failed (exit %d) - continuing with the remaining variants"
              % (label, code))

print()
print("=" * 72)
for _t, _name, _state, _n in _summary:
    _p = GPU_PEAKS.get(_name) or {}
    _gpu = " | ".join("GPU %d mean %d%% peak %d%%"
                      % (i, _p[i]["sum_util"] // max(1, _p[i]["n"]),
                         _p[i]["max_util"]) for i in sorted(_p))
    print("  %-9s %-40s %-12s %d .cif  %s" % (_t, _name, _state, _n, _gpu))
print("=" * 72)
if any(s[2].startswith("FAILED") for s in _summary):
    print("Re-running this cell retries only the failed variants.")



== RUN   template=none -> ./output/egfr-rad51-monomer-afatinib
   opendde pred -i /content/OpenDDE/egfr-rad51-monomer-afatinib.json --use_template false -o ./output -n opendde_v1 --use_msa true --use_rna_msa false --sample 1 --step 200 --cycle 10 --need_atom_confidence true --dtype bf16



In [16]:
# @title Download every output as one zip
# Archives everything under ./output - every run, templated or not - into a
# single zip and downloads it, instead of the cell above which zips only the
# variant currently selected.
#
# The archive is written to /content, deliberately outside ./output so it cannot
# end up inside itself, and that is also where the runtime-delete cell looks for
# it. That cell understands this combined archive: a run counts as safe if a zip
# contains its directory, not only if a <run_name>.zip exists.
#
# Leave ARCHIVE_NAME empty for a timestamped name, which avoids silently
# overwriting an earlier archive.
ARCHIVE_NAME = "egfr-rad51-monomer-afatinib"  # @param {type:"string"}

import datetime
import glob
import os
import shutil

OUT_ROOT = "./output"
ARCHIVE_DIR = "/content"
WARN_BYTES = 2_000_000_000  # Colab's browser download gets unreliable up here


def _tree_size(path):
    total = 0
    for root, _dirs, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return total


run_dirs = sorted(d for d in glob.glob(os.path.join(OUT_ROOT, "*"))
                  if os.path.isdir(d))
if not run_dirs:
    print("Nothing under %s yet - run a prediction first." % OUT_ROOT)
else:
    print("%-52s %-7s %-7s %s" % ("run", ".cif", ".json", "size"))
    print("-" * 82)
    grand = 0
    for d in run_dirs:
        n_cif = len(glob.glob(os.path.join(d, "**", "*.cif"), recursive=True))
        n_json = len(glob.glob(os.path.join(d, "**", "*.json"), recursive=True))
        size = _tree_size(d)
        grand += size
        print("%-52s %-7d %-7d %.1f MB"
              % (os.path.basename(d), n_cif, n_json, size / 1e6))
    print("-" * 82)
    print("%d run(s), %.2f GB uncompressed" % (len(run_dirs), grand / 1e9))

    stem = ARCHIVE_NAME.strip() or (
        "opendde-output-all-" + datetime.datetime.now().strftime("%Y%m%d-%H%M"))
    if stem.lower().endswith(".zip"):
        stem = stem[:-4]
    base = os.path.join(ARCHIVE_DIR, os.path.basename(stem))

    print()
    print("Archiving %s -> %s.zip  (this takes a while for several GB)"
          % (OUT_ROOT, base))
    archive = shutil.make_archive(base, "zip", root_dir=OUT_ROOT)
    size = os.path.getsize(archive)
    print("Wrote %s, %.2f GB (%.0f%% of uncompressed)"
          % (archive, size / 1e9, 100.0 * size / max(1, grand)))

    if size > WARN_BYTES:
        print()
        print("That is over %.1f GB. Browser downloads this large often stall or "
              "fail part way; the file is already on disk at %s, so a failed "
              "download can be retried without rebuilding it. The sturdier route "
              "is Google Drive:" % (WARN_BYTES / 1e9, archive))
        print("    from google.colab import drive")
        print("    drive.mount('/content/drive')")
        print("    import shutil; shutil.copy(%r, '/content/drive/MyDrive/')"
              % archive)
        print("Note the runtime-delete cell counts this archive as protecting "
              "every run inside it, so verify the download before deleting.")

    print()
    try:
        from google.colab import files
        files.download(archive)
    except ImportError as exc:
        print("Not on Colab (%s) - the archive is at %s" % (exc, archive))

run                                                  .cif    .json   size
----------------------------------------------------------------------------------
egfr-rad51-monomer-afatinib                          25      51      402.7 MB
egfr-rad51-monomer-afatinib-tactive                  25      51      402.7 MB
egfr-rad51-monomer-afatinib-tinactive                25      51      402.7 MB
----------------------------------------------------------------------------------
3 run(s), 1.21 GB uncompressed

Archiving ./output -> /content/egfr-rad51-monomer-afatinib.zip  (this takes a while for several GB)
Wrote /content/egfr-rad51-monomer-afatinib.zip, 0.15 GB (13% of uncompressed)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Post-processing: visualize structures + rank by ipSAE

Requires the `pred` runs above to have used `--need_atom_confidence true` (adds `*_full_data_sample_*.json` per model, with the PAE matrix and per-atom pLDDT).

In [ ]:
# @title Which template variant should the cells below read?
# The sweep writes three output directories; every analysis cell below reads
# ./output/<PROJECT_NAME>, so this is where you choose between them. Re-run
# this cell and the ones after it to switch variants.
ANALYZE_TEMPLATE = "none"  # @param ["none", "active", "inactive"]

if "RUNS_BY_TEMPLATE" not in globals():
    raise RuntimeError("run the template sweep cell above first")
PROJECT_NAME = RUNS_BY_TEMPLATE[ANALYZE_TEMPLATE]["run_name"]
print("analysis cells below will read ./output/%s" % PROJECT_NAME)


In [ ]:
# @title Shared helpers: locate each model's .cif plus its matching per-sample JSON
# sidecars, following OpenDDE's dumper.py naming convention:
#   {name}_sample_{rank}.cif
#   {name}_summary_confidence_sample_{rank}.json   (always written)
#   {name}_full_data_sample_{rank}.json             (only with --need_atom_confidence true)
import glob
import os
import re

_SAMPLE_RE = re.compile(r"(.+)_sample_(\d+)\.cif$")


def find_all_cifs(output_dir="./output/{}".format(PROJECT_NAME)):
    return sorted(glob.glob(os.path.join(output_dir, "**", "*.cif"), recursive=True))


def _sample_stem_rank(cif_path):
    m = _SAMPLE_RE.match(os.path.basename(cif_path))
    return m.groups() if m else (None, None)


def find_full_data_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_full_data_sample_{rank}.json")
    return p if os.path.exists(p) else None


def find_summary_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_summary_confidence_sample_{rank}.json")
    return p if os.path.exists(p) else None


def model_label(cif_path):
    # Short "seed N, sample M" label instead of the full relative path.
    _, rank = _sample_stem_rank(cif_path)
    seed_match = re.search(r"seed_(\d+)", cif_path)
    seed = seed_match.group(1) if seed_match else "?"
    return f"seed {seed}, sample {rank}" if rank is not None else os.path.basename(cif_path)


cifs = find_all_cifs()
print(f"Found {len(cifs)} predicted structure(s) under ./output/{PROJECT_NAME}")

In [ ]:
#@title Visualize predicted structures + PAE (py2Dmol)
!pip install -q py3Dmol py2Dmol

import json

import numpy as np
import py2Dmol

load_as_frames = True  # @param {type:"boolean"}
viewer_size = 400


def load_pae(cif_path):
    # OpenDDE's full_data.json stores the PAE matrix as "token_pair_pae"
    # (AlphaFold3's equivalent full_data.json key is "pae").
    fd_path = find_full_data_json(cif_path)
    if fd_path is None:
        return None
    with open(fd_path) as f:
        data = json.load(f)
    pae = data.get("token_pair_pae")
    return np.asarray(pae, dtype=float) if pae is not None else None


if not cifs:
    print("No .cif files found under ./output yet - run a prediction first.")
else:
    viewer = py2Dmol.view(size=(viewer_size, viewer_size), pae=True, autoplay=load_as_frames)
    for i, cif in enumerate(cifs, start=1):
        pae = load_pae(cif)
        if pae is None:
            print(f"No PAE data for {cif} (re-run pred with --need_atom_confidence true)")
        name = "models" if load_as_frames else os.path.basename(cif).replace(".cif", "")
        viewer.add_pdb(cif, name=name, paes=pae)
    viewer.show()

No .cif files found under ./output yet - run a prediction first.


## Utilities

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("{}".format(PROJECT_NAME), "zip", "./output/{}".format(PROJECT_NAME))
files.download("{}.zip".format(PROJECT_NAME))

In [ ]:
import glob, os

for pattern in ("*.txt", "*.pml"):
    for f in glob.glob(f"./output/**/{pattern}", recursive=True):
        os.remove(f)

In [ ]:
# @title Disconnect and delete the runtime
# IRREVERSIBLE. Deleting the runtime wipes this machine's disk: ./output, every
# .zip still sitting here, the staged templates, and the ~90 GB of downloaded
# model/common data. Nothing is recoverable afterwards. The upside is that the
# GPU is released immediately instead of idling against your quota.
#
# The sweep writes one output directory PER TEMPLATE VARIANT while the download
# cell above zips only the variant currently selected, so the easiest way to
# lose work here is to download one variant and delete the runtime with the
# other two still on disk. This cell therefore audits every run first and
# refuses to delete anything that has .cif files but no .zip beside them.
#
# A .zip on disk only proves you zipped it, not that the browser finished
# saving it - check your own Downloads folder before ticking the box.
I_HAVE_DOWNLOADED_THE_RESULTS = False  # @param {type:"boolean"}
DELETE_EVEN_IF_ZIPS_ARE_MISSING = False  # @param {type:"boolean"}

import glob
import os
import zipfile

# The install cell does `%cd OpenDDE`, so make_archive in the per-run
# download cell writes to /content/OpenDDE, while the download-everything
# cell writes to /content; check the plausible places regardless.
ZIP_DIRS = [os.getcwd(), "/content", "/content/OpenDDE"]


def _runs_inside_zips():
    """{run_name: zip_path} for runs a combined archive already contains.

    The download-everything cell archives the CONTENTS of ./output, so its
    top-level entries are the run directories. A per-run zip instead holds
    that run's seed_* directories, and is matched by filename below.
    """
    cover, seen = {}, set()
    for d in ZIP_DIRS:
        for z in sorted(glob.glob(os.path.join(d, "*.zip"))):
            real = os.path.realpath(z)
            if real in seen:
                continue
            seen.add(real)
            try:
                with zipfile.ZipFile(z) as zf:
                    tops = {n.split("/")[0] for n in zf.namelist() if "/" in n}
            except Exception:
                continue  # unreadable or not a zip: it protects nothing
            for t in tops:
                cover.setdefault(t, z)
    return cover

if "RUNS" in globals() and RUNS:
    run_names = [r["run_name"] for r in RUNS]
else:
    # Fall back to whatever is on disk, so this works without the sweep cell.
    run_names = sorted(os.path.basename(p.rstrip("/\\"))
                       for p in glob.glob("./output/*") if os.path.isdir(p))

_combined = _runs_inside_zips()
audit = []
for name in run_names:
    n_cif = len(glob.glob(os.path.join("./output", name, "**", "*.cif"), recursive=True))
    zpath = next((os.path.join(d, name + ".zip") for d in ZIP_DIRS
                  if os.path.isfile(os.path.join(d, name + ".zip"))), None)
    audit.append((name, n_cif, zpath or _combined.get(name)))

print("%-44s %-9s %s" % ("run", ".cif", "zip on disk"))
print("-" * 92)
for name, n_cif, zpath in audit:
    size = "%.1f MB" % (os.path.getsize(zpath) / 1e6) if zpath else ""
    print("%-44s %-9d %s %s" % (name, n_cif, zpath or "- none -", size))
print("-" * 92)

at_risk = [name for name, n_cif, zpath in audit if n_cif and not zpath]
empty = [name for name, n_cif, _ in audit if not n_cif]
if empty:
    print("No .cif files (nothing to lose): %s" % ", ".join(empty))

if not audit:
    print()
    print("No runs found under ./output - nothing to protect.")

if not I_HAVE_DOWNLOADED_THE_RESULTS:
    print()
    print("Nothing deleted. The runtime is still running and still consuming quota.")
    print("Download what you need, then tick I_HAVE_DOWNLOADED_THE_RESULTS and "
          "re-run this cell.")
    if at_risk:
        print("Not yet zipped: %s" % ", ".join(at_risk))
elif at_risk and not DELETE_EVEN_IF_ZIPS_ARE_MISSING:
    raise RuntimeError(
        "Refusing to delete the runtime: these runs have .cif files but no .zip "
        "on disk, so their results exist nowhere else:\n  "
        + "\n  ".join(at_risk)
        + "\n\nSelect each one in the analysis selector cell and run the download "
          "cell for it, or tick DELETE_EVEN_IF_ZIPS_ARE_MISSING to discard them "
          "on purpose.")
else:
    if at_risk:
        print()
        print("Discarding un-zipped results on purpose: %s" % ", ".join(at_risk))
    print()
    print("Unassigning the runtime now. The notebook will disconnect; its disk, "
          "including ./output, is gone.")
    try:
        from google.colab import runtime
    except ImportError as exc:
        raise RuntimeError(
            "google.colab.runtime is unavailable (%s) - this cell only does "
            "anything on Colab. Stop the kernel yourself elsewhere." % exc)
    runtime.unassign()